<center>
  <h3>More than a Hack 2027 &middot; Phase 1</h3>
  <h2>Frame-level Ball Possession &mdash; E01 Baseline</h2>

  <p><strong>Metrica Sports &times; FC Barcelona Innovation Hub &times; Mobile World Capital</strong></p>
</center>

---

### Carried By Admin Team - Sepuluh Nopember Institute of Technology - Indonesia

- **Rasya Arya Ramadhan**
- **Benedictus Ryu Gunawan**
- **Ahmad Naufal Farras**
- **Jeremy Mattathis Mboe**
- **Jylan Annisa Mumtaza Syidana**

---

## Summary

The official baseline: the player nearest to the ball is taken to have possession, `D` when the ball is not visible, then a **majority vote** over a window of ±*w* frames. The four variants differ only in *w* ∈ {0, 25, 50, 125}.

The goal of this experiment is to reproduce the published floor (`baseline_smooth50`, spellF1 0.2433 on the full match) and to understand why `macroF1` and `spellF1` disagree.

# Methodology

**ATD tracking → player nearest to the ball (metres) → team code per frame → majority vote over a ±*w* window → `out/*.csv` → validation → score on the public GT.**

# Environment

Run from inside the repository. Parsing the ATD takes about a minute on the first run and is then cached.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
for p in (ROOT, ROOT / "starter", ROOT / "evaluation", ROOT / "experiments"):
    sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import common
from games import asset_paths, half_video_blocks, source_frames

FPS = 25
N_FRAMES = source_frames("tactical")
BLOCKS = half_video_blocks("BAR-ATH")   # [(half, first, last)] -> graded frames
GRADED = np.zeros(N_FRAMES, bool)
for _, a, b in BLOCKS:
    GRADED[a:b + 1] = True
COLORS = {"H": "#A50044", "A": "#C8102E", "D": "#7F7F7F", "X": "#DDDDDD"}
print("ROOT:", ROOT, "| frames:", N_FRAMES, "| graded:", GRADED.sum())

# Approach: Nearest Player + Majority Vote

## [1] Team code per frame

ATD coordinates are normalised to 0–1 and the pitch is not square, so both axes are converted to metres before distances are compared.

In [ ]:
from baseline import build, smooth, write_submission
codes = common.cached("atd_nearest", build)       # 0=H, 1=A, -1=no ball
print(pd.Series(codes[GRADED]).map({0: "H", 1: "A", -1: "no ball"}).value_counts().to_string())

## [2] Smoothing with different windows

Without smoothing the labels flicker between frames, so the number of possessions explodes.

In [ ]:
OUT = Path(common.HERE) / "E01_baseline" / "out"
variants = {"smooth%d" % w: smooth(codes, w) for w in (0, 25, 50, 125)}
for name, labels in variants.items():
    write_submission(str(OUT / (name + ".csv")), labels)
print(sorted(p.name for p in OUT.glob("*.csv")))

## [3] Score on the public GT

A **pipeline check** only. The numbers below use 17 possessions, so small differences are not meaningful.

In [ ]:
import io, contextlib, subprocess
res = subprocess.run([sys.executable, str(ROOT / "evaluation" / "run_evaluation.py"),
                      "--results", str(OUT), "--out", str(OUT / "reports")],
                     capture_output=True, text=True)
board = pd.read_csv(OUT / "reports" / "scoreboard.csv")
display(board)

## [4] Public vs full match

The full-match scores below are quoted from `EVALUATION.md` (not recomputed here, because the full GT is private).

In [ ]:
full = pd.DataFrame({"variant": ["raw", "smooth25", "smooth50", "smooth125"],
                     "macroF1_full": [0.6532, 0.6752, 0.6724, 0.6356],
                     "spellF1_full": [0.0486, 0.1996, 0.2433, 0.2242]})
display(full)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].bar(full.variantt, full.macroF1_full, color="#004D98"); ax[0].set_title("macroF1 (full match)")
ax[1].bar(full.variantt, full.spellF1_full, color="#A50044"); ax[1].set_title("spellF1 (full match)")
plt.tight_layout(); plt.show()

`macroF1` moves by ±0.04 and picks `smooth25`, while `spellF1` moves five times as much and picks `smooth50`. On the public GT, `smooth125` is actually best; choosing the window from the public 5 minutes would be wrong.

## [5] Output

The baseline submissions are in `experiments/E01_baseline/out/`. Later experiments must beat spellF1 0.2433 on the full match without tuning on the public GT.